# Phase 2 — FastMCP server + trivial tool, proven end to end

Items 1+2 (`docs/build-order.md`): stand up a real FastMCP server, then
prove it works by having a real LLM call a trivial tool *through* it, via a
real LangGraph.

The server runs in a background thread since `mcp.run(...)` blocks forever
— same reason any long-running server can't just be called in a cell
directly.

In [1]:
import os
import sys

sys.path.insert(0, "..")  # so `app.*` resolves from notebooks/'s cwd

from app.config import GCP_PROJECT_ID, get_secret

os.environ["ANTHROPIC_API_KEY"] = get_secret("anthropic-api-key", GCP_PROJECT_ID)
print("key loaded")

key loaded


## The server — bare FastMCP instance, one trivial tool

In [ ]:
import threading
import time

from fastmcp import FastMCP

MCP_PORT = 8765

mcp = FastMCP("analytics-test")


@mcp.tool()
def ping(message: str) -> str:
    """Echoes the given message back, prefixed with 'pong: '. A trivial
    proof tool, not a real one — confirms the MCP round-trip works."""
    return f"pong: {message}"


def _run_server():
    # "http", not "streamable_http" — confirmed against the official
    # langchain-mcp-adapters README example (2026-09-18), which pairs this
    # exact server-side value with the client's "http" transport below.
    mcp.run(transport="http", port=MCP_PORT)


server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()
time.sleep(1)
print(f"MCP server started on port {MCP_PORT}")

╭──────────────────────────────────────────────────────────────────────────────╮                  
                 │                                                                              │                  
                 │                                                                              │                  
                 │                         ▄▀▀ ▄▀█ █▀▀ ▀█▀ █▀▄▀█ █▀▀ █▀█                        │                  
                 │                         █▀  █▀█ ▄▄█  █  █ ▀ █ █▄▄ █▀▀                        │                  
                 │                                                                              │                  
                 │                                                                              │                  
                 │                                                                              │                  
                 │                                FastMCP 4.0.3                                 │                  
                 │                            https://gofastmcp.com                             │                  
                 │                                                                              │                  
                 │                  🖥  Server:      analytics-test, 4.0.3                       │                  
                 │                  🚀 Deploy free: https://horizon.prefect.io                  │                  
                 │                                                                              │                  
                 ╰──────────────────────────────────────────────────────────────────────────────╯                  
                 ╭──────────────────────────────────────────────────────────────────────────────╮                  
                 │                          🎉 Update available: 4.0.5                          │                  
                 │                      Run: pip install --upgrade fastmcp                      │                  
                 ╰──────────────────────────────────────────────────────────────────────────────╯

[09/18/26 20:23:22] INFO     Starting MCP server 'analytics-test' with transport 'http' on         ]8;id=11389317;file://c:\Users\jmhla\projects\analytics-agent\.venv\Lib\site-packages\fastmcp\server\mixins\transport.py\transport.py]8;;\:]8;id=11389318;file://c:\Users\jmhla\projects\analytics-agent\.venv\Lib\site-packages\fastmcp\server\mixins\transport.py#363\363]8;;\
                             http://127.0.0.1:8765/mcp                                                             

MCP server started on port 8765


INFO:     Started server process [22272]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8765 (Press CTRL+C to quit)


## The client — discover tools over the real MCP protocol


In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_client = MultiServerMCPClient({
    "analytics": {"transport": "http", "url": f"http://localhost:{MCP_PORT}/mcp"},
})
tools = await mcp_client.get_tools()
print(f"{len(tools)} tool(s) discovered")
for t in tools:
    print(f"- {t.name}: {t.description}")

ImportError: cannot import name 'RequestContext' from 'mcp.shared.context' (c:\Users\jmhla\projects\analytics-agent\.venv\Lib\site-packages\mcp\shared\context.py)

## A minimal LangGraph — agent node + tool-execution node

Genuinely two nodes, not one: an "agent" call alone would only show the LLM
*deciding* to call the tool, not it actually being executed through MCP.
`ToolNode` is what actually runs the call and feeds the real result back.

In [ ]:
from typing import Annotated, TypedDict

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode


class State(TypedDict):
    messages: Annotated[list, add_messages]


llm = ChatAnthropic(model="claude-sonnet-5").bind_tools(tools)


async def agent_node(state: State) -> dict:
    response = await llm.ainvoke(state["messages"])
    return {"messages": [response]}


def route(state: State):
    last = state["messages"][-1]
    return "tools" if getattr(last, "tool_calls", None) else END


g = StateGraph(State)
g.add_node("agent", agent_node)
g.add_node("tools", ToolNode(tools))
g.add_edge(START, "agent")
g.add_conditional_edges("agent", route, {"tools": "tools", END: END})
g.add_edge("tools", "agent")
graph = g.compile()
print("graph compiled")

## Run it — confirm the tool actually gets called, not just requested

In [ ]:
result = await graph.ainvoke({
    "messages": [HumanMessage(content="Use the ping tool to say hello")]
})

for m in result["messages"]:
    tool_calls = getattr(m, "tool_calls", None)
    print(f"{type(m).__name__}: {m.content!r}" + (f"  tool_calls={tool_calls}" if tool_calls else ""))